Copyright 2026 Google LLC.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Priority and Flex Inference Tiers

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Inference_tiers.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

The Gemini API offers different `service_tiers` to help you manage cost and latency.

The **Priority** and **Flex** tiers allow you to route background jobs to Flex and interactive jobs to Priority using standard synchronous endpoints. This eliminates the complexity of async job management while giving you the economic and performance benefits of specialized tiers:

- **[Priority](https://ai.google.dev/gemini-api/docs/priority-inference) (`"priority"`):** Millisecond latency for critical apps (+75-100% cost). Traffic is strictly non-sheddable.
- **[Flex](https://ai.google.dev/gemini-api/docs/flex-inference) (`"flex"`):** 1-15 min target latency for background tasks (-50% cost). Fully synchronous, but uses opportunistic compute.

**In this notebook, you will learn:**
1.  Comparison of Priority and Flex
2.  How to use Priority tier
3.  How to use Flex tier
4.  How to adjust timeouts
5.  How to implement retries

> **Note:** Inference tiers are paid only features. Flex is available for all tiers on the paid tier, and Priority is available for Tier 2 & 3. This notebook won't run with the Free Tier. (cf. [pricing](https://ai.google.dev/pricing) for more details).

# Setup

### Install SDK

Install the SDK from [PyPI](https://github.com/googleapis/python-genai).

In [ ]:
%pip install -U -q "google-genai>=1.70.0"  # Minimum version 1.70 for service tiers

### Setup your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) for an example.

In [2]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

### Initialize SDK client

Initialize a client with your API key:

In [5]:
from google import genai
from google.genai import errors, types

client = genai.Client(api_key=GEMINI_API_KEY)

### Choose a model

Most Gemini 2.5 and Gemini 3 models support inference tiers. Refer to the documentation for more details:
- [Flex supported models](https://ai.google.dev/gemini-api/docs/flex-inference#supported-models)
- [Priority supported models](https://ai.google.dev/gemini-api/docs/priority-inference#supported-models)

In [18]:
MODEL_ID = "gemini-3-flash-preview" # @param ["gemini-3.1-flash-lite-preview", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

## Priority and Flex comparison

| Feature | Standard | Flex | Priority | Batch | Caching |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Pricing** | Full Price | 50% discount | 75% to 100% more than standard | 50% discount | 90% discount + Prorated token storage |
| **Latency** | Seconds to minutes | Minutes (1–15 min target) | Seconds | Up to 24 hours | Faster time-to-first-token |
| **Reliability** | High / Medium-high | Best-effort (Sheddable) | High (Non-sheddable) | High (for throughput) | N/A |
| **Interface** | Synchronous | Synchronous | Synchronous | Asynchronous | Saved state |
| **Best use case** | General application workflows | Non-urgent sequential chains | Production, user-facing apps | Massive datasets, offline evals | Recurring queries over same file |

### Key benefits of Priority

* **Low latency**: Designed for second response times for interactive,
user-facing AI tools.
* **High reliability**: Traffic is treated with the highest criticality and is
strictly non-sheddable.
* **Graceful degradation**: Traffic spikes exceeding dynamic limits are
automatically downgraded to the Standard tier for processing instead of failing,
preventing service outages.

### Key benefits of Flex

* **Cost efficiency**: Substantial savings for non-production evals, background agents, and data enrichment.
* **Low friction**: No need to manage batch objects, job IDs, or polling; simply add a single parameter to your existing requests.
* **Synchronous workflows**: Ideal for sequential API chains where the next request depends on the output of the previous one, making it more flexible than Batch for agentic workflows.

## Priority inference

The Gemini Priority API is a premium inference tier designed for
business-critical workloads that require lower latency and the highest
reliability at a premium price point. Priority tier traffic is prioritized above
standard API and Flex tier traffic.

### How Priority works

Priority inference routes requests to high-criticality compute queues, offering
predictable, fast performance for user-facing applications. Its primary
mechanism is a graceful server-side downgrade to standard processing for traffic
that exceeds dynamic limits, ensuring application stability instead of failing
the request.

### Priority rate limits

Priority consumption holds its own rate
limits even though consumption is counted towards overall interactive traffic
rate limits. **Default rate limits are: 0.3x the [standard rate limit](https://aistudio.google.com/rate-limit) for each model and tier**.

### How to use Priority

To use the Priority tier, set the `service_tier` field in the request body to
`"priority"`. The default tier is standard if the field is omitted.

In [19]:
try:
    response = client.models.generate_content(
        model=MODEL_ID,
        contents="Why is the sky blue?",
        config={'service_tier': 'priority'},
    )

    # Validate for graceful downgrade
    if response.sdk_http_response.headers.get('x-gemini-service-tier') == "standard":
        print("Warning: Priority limit exceeded, processed at Standard tier.")

    print(response.text)

except errors.APIError as e:
    print(e.code, e.message)
except Exception as e:
    print(f"Error during API call: {e}")

The sky appears blue because of a phenomenon called **Rayleigh scattering**.

Here is the step-by-step breakdown of how it works:

### 1. Sunlight is a mix of all colors
Although sunlight looks white to us, it is actually composed of all the colors of the rainbow (red, orange, yellow, green, blue, indigo, and violet). Light travels as waves, and each color has a different wavelength:
*   **Red light** has long, lazy wavelengths.
*   **Blue and violet light** have short, choppy wavelengths.

### 2. The atmosphere acts like an obstacle course
Earth’s atmosphere is filled with gases (mostly nitrogen and oxygen) and tiny particles. When sunlight hits the atmosphere, it strikes these gas molecules.

### 3. Scattering occurs
Because blue light travels in shorter, smaller waves, it crashes into the gas molecules more frequently than the longer red waves do. When the blue light hits these molecules, it gets scattered in every direction. 

As you look up, your eyes detect this scattered blue li

You'll find the service tier in the headers:

In [20]:
print(response.sdk_http_response.headers.get('x-gemini-service-tier'))

priority


## Flex inference

The Gemini Flex API is an inference tier that offers a 50% cost reduction
compared to standard rates, in exchange for variable latency and best-effort
availability. It's designed for latency-tolerant workloads that require
synchronous processing but don't need the real-time performance of the standard
API.

### How Flex works

Gemini Flex inference bridges the gap between the standard API and the 24-hour
turnaround of the [Batch API](https://ai.google.dev/gemini-api/docs/batch-api). It utilizes off-peak,
"sheddable" compute capacity to provide a cost-effective solution for background
tasks and sequential workflows.

### How to use Flex

To use the Flex tier, specify the `service_tier` as `"flex"` in the
request body. By default, requests use the standard tier if this field is
omitted.


In [21]:
try:
    response = client.models.generate_content(
        model=MODEL_ID,
        contents="Why is the sky blue?",
        config={'service_tier': 'flex'},
    )

    print(response.text)
except errors.APIError as e:
    print(e.code, e.message)

The short answer is that the sky is blue because of a phenomenon called **Rayleigh scattering**.

Here is the step-by-step breakdown of why it happens:

### 1. Sunlight is made of all colors
Although sunlight looks white to us, it is actually composed of all the colors of the rainbow (red, orange, yellow, green, blue, indigo, and violet). Light travels as waves, and each color has a different wavelength:
*   **Red light** has long, lazy wavelengths.
*   **Blue and violet light** have short, choppy wavelengths.

### 2. The Atmosphere is an obstacle course
Earth’s atmosphere is filled with gases (mostly nitrogen and oxygen). As sunlight reaches the atmosphere, it strikes the molecules of these gases and scatters in every direction.

### 3. Blue light scatters the most
Because blue light travels in shorter, smaller waves, it hits the gas molecules more frequently and is scattered much more strongly than the other colors. This "scattered" blue light is what your eyes see coming from every 

In [22]:
print(response.sdk_http_response.headers.get('x-gemini-service-tier'))

flex


### Adjusting timeout windows

You can configure per-request timeouts for the REST API and client libraries,
and global timeouts only when using the client libraries.

Always ensure your client-side timeout covers the intended server patience
window (e.g., 600s+ for Flex wait queues). The SDKs expect timeout values in
milliseconds.

#### Per-request timeouts


In [24]:
try:
    response = client.models.generate_content(
        model=MODEL_ID,
        contents="why is the sky blue?",
        config={
            "service_tier": "flex",
            "http_options": {"timeout": 900000}
        },
    )
    print(response.text)
except errors.APIError as e:
    print(e.code, e.message)

The sky appears blue due to a phenomenon called **Rayleigh scattering**. 

Here is the step-by-step breakdown of how it works:

### 1. Sunlight is a mix of all colors
Although sunlight looks white to us, it is actually made up of all the colors of the rainbow (red, orange, yellow, green, blue, indigo, and violet). Light travels as waves, and each color has a different wavelength:
*   **Red light** has long, lazy, stretched-out wavelengths.
*   **Blue and violet light** have short, choppy, frequent wavelengths.

### 2. The Atmosphere acts like an obstacle course
Earth’s atmosphere is filled with gases (mostly nitrogen and oxygen). When sunlight hits the atmosphere, it collides with these gas molecules.

### 3. Scattering happens
Because blue light travels in shorter, smaller waves, it is much more likely to strike these gas molecules and get **scattered** in every direction. Red and yellow light, with their longer wavelengths, pass through the atmosphere mostly uninterrupted.

So, when 

Example with streaming (timeout applies to each chunk or overall, depending on implementation)

In [25]:
try:
    response = client.models.generate_content_stream(
        model=MODEL_ID,
        contents=["List 5 ideas for a sci-fi movie."],
        config={
            "service_tier": "flex",
            "http_options": {"timeout": 60000}
        },
        # Per-request timeout for the streaming operation
    )
    for chunk in response:
        print(chunk.text, end="")
except errors.APIError as e:
    print(e.code, e.message)
except Exception as e:
    print(f"An error occurred during streaming: {e}")

Here are five original sci-fi movie concepts, ranging from psychological thrillers to high-concept space operas:

### 1. The Echo Archive
**The Concept:** In a future where memories can be digitised and "donated" to a collective global library, a professional "Memory Editor" discovers a series of memories that shouldn’t exist. They depict a secret history of the world that conflicts with official records. 
**The Hook:** To uncover the truth, the protagonist must navigate the fractured consciousness of strangers, but they soon realize that every time they access a forbidden memory, a piece of their own personality is permanently overwritten.

### 2. Orbital Decay
**The Concept:** Earth has become a prison planet, entirely encased in a sophisticated, automated satellite grid that prevents anything from leaving or entering. A low-level technician tasked with maintaining the "Kill-Switch" satellites discovers that the grid isn't keeping the world's threats *in*—it’s protecting the planet f

#### Global timeouts

If you want all API calls made through a specific `genai.Client` instance
(client libraries only) to have a default timeout, you can configure this when
initializing the client using `http_options` and `genai.types.HttpOptions`.

In [27]:
global_timeout_ms = 120000

client_with_global_timeout = genai.Client(
    api_key=GEMINI_API_KEY,
    http_options=types.HttpOptions(timeout=global_timeout_ms)
)

try:
    # Calling generate_content using global timeout...
    response = client_with_global_timeout.models.generate_content(
        model=MODEL_ID,
        contents="Summarize the history of AI development since 2000.",
        config={"service_tier": "flex"},
    )
    print(response.text)

    # A per-request timeout will *override* the global timeout for that specific call.
    shorter_timeout = 30000
    response = client_with_global_timeout.models.generate_content(
        model=MODEL_ID,
        contents="Provide a very brief definition of machine learning.",
        config={
            "service_tier": "flex",
            "http_options":{"timeout": shorter_timeout}
        }  # Overrides the global timeout
    )

    print(response.text)

except TimeoutError:
    print(
        f"A GenerateContent call timed out. Check if the global or per-request timeout was exceeded."
    )
except errors.APIError as e:
    print(e.code, e.message)
except Exception as e:
    print(f"An error occurred: {e}")

The history of AI since 2000 is a transition from symbolic, rule-based systems to probabilistic, data-driven "deep learning." This evolution can be broken down into four distinct eras.

### 1. The Era of Foundation and Narrow AI (2000–2010)
At the turn of the millennium, AI research was focused on specialized tasks rather than general intelligence.
*   **The Rise of Machine Learning:** Researchers moved away from "expert systems" (where humans hard-coded rules) toward systems that learned from data. Statistical methods, such as Support Vector Machines (SVMs), dominated.
*   **Milestones:** In 2005, a Stanford robot won the DARPA Grand Challenge by navigating a desert course, proving that autonomous navigation was possible. 
*   **The Big Data Catalyst:** The explosion of the internet began providing the massive datasets required to train more sophisticated models.

### 2. The Deep Learning Breakthrough (2011–2015)
This period marked the most significant shift in AI history, triggered b

### Implementing retries

Because Flex is sheddable and fails with 503 errors, here is an example of
optionally implementing retry logic to continue with failed requests:

In [29]:
import time


def call_with_retry(max_retries=3, base_delay=5):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=MODEL_ID,
                contents="Provide a very brief definition of machine learning.",
                config={"service_tier": "flex"},
            )
        except errors.APIError as e:
            # Check for 503 Service Unavailable or 429 Rate Limits
            print(e.code)
            if attempt < max_retries - 1:
                delay = base_delay * (2**attempt)  # Exponential Backoff
                print(f"Flex busy, retrying in {delay}s...")
                time.sleep(delay)
            else:
                # Fallback to standard on last strike
                print("Flex exhausted, falling back to Standard...")
                return client.models.generate_content(
                    model=MODEL_ID,
                    contents="Provide a very brief definition of machine learning.",
                )
        except Exception as e:
            print(f"An error occurred: {e}")

# Usage
response = call_with_retry()
print(response.text)


Machine learning is a field of artificial intelligence that enables computers to learn from data and improve at tasks without being explicitly programmed for them.


## Further resources

To learn more, see the following resources:

- [Optimization docs](https://ai.google.dev/gemini-api/docs/optimization)
- [Priority docs](https://ai.google.dev/gemini-api/docs/priority-inference)
- [Flex docs](https://ai.google.dev/gemini-api/docs/flex-inference)